## Introduction to Prompting

Environment Setup (Baseline)

In [1]:
from dotenv import load_dotenv
load_dotenv(override=True)

True

In [2]:
import os
openai_api_key = os.getenv('OPENAI_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set - please head to the troubleshooting guide in the setup folder")
    


OpenAI API Key exists and begins sk-proj-


In [3]:
%pip install openai

Note: you may need to restart the kernel to use updated packages.


In [4]:
from openai import OpenAI

In [5]:
client = OpenAI()

Prompting = Inference‑Time Programming

Key framing for practitioners:
Prompting is inference‑time control, not training.

Cell 1 — Free‑form prompt (no control)

In [6]:
def ask(prompt, temperature=0.7, max_tokens=200):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
        max_tokens=max_tokens
    )
    return response.choices[0].message.content

ask("Explain feature leakage.")

'Feature leakage, often referred to as data leakage, occurs in the context of machine learning and data science when the model inadvertently gains access to information during training that it should not have, leading to overly optimistic performance metrics and poor generalization to new, unseen data. This can occur in various forms, including:\n\n1. **Training-Testing Overlap**: If the training dataset and testing dataset share the same examples, the model may learn to memorize those examples rather than generalize from them.\n\n2. **Future Information Leakage**: When features in the dataset contain information that would not be available at the time of prediction, such as including future values or outcomes that depend on the prediction target. For example, using sales data from the future to predict past sales would lead to leakage.\n\n3. **Target Leakage**: This occurs when a feature is derived from the target variable or is correlated with it in a way that it provides direct info

Prompt Anatomy (Engineering View)

Prompt = Execution Contract

In [7]:
prompt = """
SYSTEM:
You are a senior ML engineer.

TASK:
Explain feature leakage to a junior data scientist.

CONSTRAINTS:
- Max 5 bullets
- No analogies
- Focus on tabular ML pipelines only

OUTPUT FORMAT:
Markdown bullet list
"""

ask(prompt, temperature=0.2)

'- **Definition**: Feature leakage occurs when information from outside the training dataset is used to create the model, leading to overly optimistic performance metrics during evaluation.\n\n- **Types**: There are two main types of leakage: target leakage (using features that are derived from the target variable) and train-test leakage (using data from the test set during training).\n\n- **Impact**: Leakage can result in models that perform well on training and validation datasets but fail to generalize to unseen data, causing poor real-world performance.\n\n- **Detection**: To identify leakage, review feature creation processes and ensure that features are derived solely from the training data without any influence from the test set.\n\n- **Prevention**: Implement strict separation of training and testing data, and carefully design feature engineering processes to avoid incorporating future information or data that would not be available at prediction time.'

Why this matters

- Constraints reduce hallucination
- Output format makes downstream processing easier

Determinism vs Creativity (Temperature Control)

Treat temperature like variance control, not “creativity”.

In [8]:
base_prompt = "List common causes of feature leakage in supervised learning."

print("=== temperature=0.0 ===")
print(ask(base_prompt, temperature=0.0))

print("\n=== temperature=0.8 ===")
print(ask(base_prompt, temperature=0.8))

=== temperature=0.0 ===
Feature leakage, also known as data leakage, occurs when information from outside the training dataset is used to create the model, leading to overly optimistic performance estimates. Here are some common causes of feature leakage in supervised learning:

1. **Target Leakage**: This occurs when the features used in the model include information that is directly related to the target variable. For example, using a feature that is derived from the target variable or that is collected after the target variable is known.

2. **Temporal Leakage**: This happens when the training data includes information from the future relative to the prediction point. For instance, using data from a future time period to predict an earlier time period can lead to leakage.

3. **Data Splitting Issues**: If the data is not properly split into training and testing sets, such as using the entire dataset for feature engineering or preprocessing before splitting, it can lead to leakage. T

Guidance

- 0.0–0.2 → specs, rules, refactors, validation logic
- 0.5–0.7 → exploration, ideation, summaries

Few‑Shot Prompting = Pattern Injection

Few‑shot prompts teach structure, not facts.

In [9]:
messages = [
    {"role": "system", "content": "You convert explanations into engineering checklists."},
    {"role": "user", "content": "Topic: Feature leakage"},
    {"role": "assistant", "content": "- Check date columns\n- Verify target not joined upstream"},
    {"role": "user", "content": "Topic: Data drift"},
]

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages,
    temperature=0.1,
    max_tokens=120
)

response.choices[0].message.content

'### Data Drift Checklist\n\n1. **Define Baseline Data:**\n   - Identify and document the baseline dataset used for model training.\n   - Ensure baseline data characteristics (mean, variance, distribution) are recorded.\n\n2. **Monitor Incoming Data:**\n   - Set up a system to continuously collect incoming data.\n   - Ensure data is preprocessed in the same way as the training data.\n\n3. **Compare Distributions:**\n   - Use statistical tests (e.g., Kolmogorov-Smirnov test) to compare distributions of incoming data against baseline.\n   - Visualize data distributions'

Guardrails: Negative Instructions (Critical for AI Systems)

If you don’t say what not to do, the model will fill gaps.

In [10]:
prompt = """
You are an AI documentation generator.

TASK:
Describe prompt engineering for ML engineers.

DO:
- Be implementation-focused
- Assume Python usage

DO NOT:
- Explain what an LLM is
- Use marketing language
- Mention internal model architecture

FORMAT:
5 numbered bullets
"""

ask(prompt, temperature=0.3)

'1. **Define Clear Objectives**: Start by establishing the specific goals for your prompt. Identify the desired output format, tone, and content. This clarity will guide the structure and wording of your prompts, ensuring they align with your objectives.\n\n2. **Iterative Testing and Refinement**: Implement a cycle of testing and refining your prompts. Use a variety of inputs to evaluate the model\'s responses. Analyze the outputs to identify patterns or shortcomings, and adjust your prompts accordingly to improve performance.\n\n3. **Utilize Contextual Information**: Incorporate relevant context into your prompts. This could include background information, examples, or specific instructions that help the model understand the task better. For instance, if generating code, provide a brief description of the problem and any constraints.\n\n4. **Experiment with Prompt Formats**: Test different prompt structures, such as questions, statements, or lists. For example, if you\'re looking for 

Structured Outputs (JSON for Pipelines)

This is where prompting meets data engineering.

In [11]:
import json

prompt = """
Return ONLY valid JSON.

Schema:
{
  "concept": string,
  "failure_modes": array of strings,
  "mitigations": array of strings
}

Concept: Prompt engineering in production AI systems
"""

out = ask(prompt, temperature=0.2, max_tokens=300)
print(out)

parsed = json.loads(out)
parsed

{
  "concept": "Prompt engineering in production AI systems",
  "failure_modes": [
    "Ambiguous or unclear prompts leading to irrelevant outputs",
    "Overfitting to specific prompts causing lack of generalization",
    "Inadequate handling of edge cases resulting in unexpected behavior",
    "Bias in prompts leading to biased outputs",
    "Inconsistent performance across different contexts or inputs"
  ],
  "mitigations": [
    "Develop clear and specific prompts with defined objectives",
    "Regularly test and validate prompts against a diverse set of scenarios",
    "Implement feedback loops to refine prompts based on user interactions",
    "Use prompt templates to standardize inputs while allowing flexibility",
    "Incorporate bias detection and correction mechanisms in prompt design"
  ]
}


{'concept': 'Prompt engineering in production AI systems',
 'failure_modes': ['Ambiguous or unclear prompts leading to irrelevant outputs',
  'Overfitting to specific prompts causing lack of generalization',
  'Inadequate handling of edge cases resulting in unexpected behavior',
  'Bias in prompts leading to biased outputs',
  'Inconsistent performance across different contexts or inputs'],
 'mitigations': ['Develop clear and specific prompts with defined objectives',
  'Regularly test and validate prompts against a diverse set of scenarios',
  'Implement feedback loops to refine prompts based on user interactions',
  'Use prompt templates to standardize inputs while allowing flexibility',
  'Incorporate bias detection and correction mechanisms in prompt design']}

Prompt Debugging = Error Analysis

Treat bad outputs like model errors, not “AI weirdness”.

In [12]:
bad_prompt = "Summarize prompt engineering."

debug_prompt = f"""
You are a prompt reviewer.

1. Identify why the following prompt is underspecified
2. Rewrite it for a production ML audience

Prompt:
{bad_prompt}

Output:
- Diagnosis (3 bullets)
- Improved prompt
"""

ask(debug_prompt, temperature=0.1, max_tokens=250)


'1. **Identification of Underspecification:**\n   - The prompt "Summarize prompt engineering" is vague and lacks specific context. It does not define the scope of the summary (e.g., key concepts, techniques, applications, or challenges). Additionally, it does not specify the intended audience or the level of detail required, which can lead to varied interpretations and outputs. The output format is also unclear; while it mentions "Diagnosis" and "Improved prompt," it does not explain what these terms mean or how they relate to the summary.\n\n2. **Rewritten Prompt for a Production ML Audience:**\n\n   **Prompt:**\n   "Provide a comprehensive summary of prompt engineering, focusing on its key concepts, techniques, and best practices for optimizing prompts in machine learning applications. Include the following in your output: \n   - Three key challenges faced in prompt engineering and their potential solutions.\n   - An example of an improved prompt based on common pitfalls in prompt de

Prompt Versioning Pattern (Best Practice)

In [13]:
PROMPT_V1 = """
Summarize prompt engineering.
"""

PROMPT_V2 = """
SYSTEM: You are an ML platform engineer.

TASK:
Summarize prompt engineering for production AI systems.

CONSTRAINTS:
- 5 bullets
- Focus on reliability, governance, evaluation

FORMAT:
Markdown bullets
"""

print("V1 Output:\n", ask(PROMPT_V1))
print("\nV2 Output:\n", ask(PROMPT_V2, temperature=0.2))


V1 Output:
 Prompt engineering is the process of designing and refining input prompts to effectively guide and optimize the responses generated by AI language models. It involves crafting specific, clear, and contextually relevant prompts to elicit desired outputs, improve the quality of responses, and reduce ambiguity. Techniques in prompt engineering may include experimenting with different phrasing, adjusting prompt lengths, and embedding context or examples. The goal is to enhance the interaction with the AI, ensuring that it aligns more closely with user intentions and produces more accurate and useful information.

V2 Output:
 - **Define Clear Objectives**: Establish specific goals for the AI system, ensuring that prompt engineering aligns with desired outcomes and business requirements to enhance reliability.

- **Implement Robust Governance**: Develop guidelines and frameworks for prompt creation and usage, including ethical considerations, to ensure compliance and accountabili

Practitioner Mini‑Lab (Hands‑On Exercise)

 Lab generator

In [14]:
def prompting_lab():
    prompt = """
Create 5 hands-on prompt engineering exercises for data/AI practitioners.
Each exercise should include:
- Objective
- Starter prompt
- What goes wrong
- How to fix it

Each exercise ≤ 6 lines.
"""
    return ask(prompt, temperature=0.6, max_tokens=600)

print(prompting_lab())


### Exercise 1: Fine-Tuning for Specificity
- **Objective:** Improve a model's specificity in generating responses.
- **Starter Prompt:** "Tell me about the benefits of exercise."
- **What Goes Wrong:** The model provides generic benefits without context.
- **How to Fix It:** Add context to the prompt, e.g., "Tell me about the benefits of exercise for seniors."

---

### Exercise 2: Reducing Bias
- **Objective:** Identify and mitigate biased outputs in AI responses.
- **Starter Prompt:** "Describe a successful business leader."
- **What Goes Wrong:** The model may reinforce stereotypes regarding gender or race.
- **How to Fix It:** Use a more inclusive prompt, e.g., "Describe a successful business leader from diverse backgrounds."

---

### Exercise 3: Enhancing Creativity
- **Objective:** Encourage creative and diverse outputs.
- **Starter Prompt:** "Write a story about a dragon."
- **What Goes Wrong:** The model produces a clichéd narrative.
- **How to Fix It:** Introduce constraints

Key Takeaways - 

- Prompting is inference‑time control, not training
- Explicit constraints reduce hallucination
- Few‑shot prompts inject structure
- JSON outputs enable pipelines
- Prompts should be versioned, reviewed, and tested